# Foundry Synthetic Data Generation

Based on [this doc](https://learn.microsoft.com/en-us/azure/foundry/observability/how-to/evaluation-dataset-synthetic?tabs=python).

In [ ]:
%pip install "azure-ai-projects>=2.4.0" azure-identity python-dotenv

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

In [4]:
# Imports
import os
import time

from azure.ai.projects.models import (
    AgentDataGenerationJobSource,
    DataGenerationJob,
    DataGenerationJobInputs,
    DataGenerationJobOutputOptions,
    DataGenerationJobScenario,
    DataGenerationModelOptions,
    DatasetDataGenerationJobOutput,
    PromptAgentDefinition,
    SimpleQnADataGenerationJobOptions,
)

from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

In [ ]:
# Generate synthetic dataset
endpoint = os.environ[
        "AZURE_AI_PROJECT_ENDPOINT"
    ]  # Sample : https://<account_name>.services.ai.azure.com/api/projects/<project_name>
MODEL_NAME = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "")  # Sample : gpt-4o-mini
poll_interval_seconds = 10

with (
    DefaultAzureCredential() as credential,
    AIProjectClient(endpoint=endpoint, credential=credential) as project_client,
    project_client.get_openai_client() as client,
):

    # 1. Reference (or create) a prompt agent whose instructions seed generation.
    agent = project_client.agents.create_version(
        agent_name="retail-agent",
        definition=PromptAgentDefinition(
            model=MODEL_NAME,
            instructions=(
                "You are a customer support assistant for Contoso Retail. "
                "Answer questions about the product catalog, loyalty program, store hours, "
                "and the return policy. If a question falls outside this scope, say you "
                "don't have that information."
            ),
        ),
    )

    # 2. Define a SimpleQnA evaluation job sourced from the agent definition.
    job = DataGenerationJob(
        inputs=DataGenerationJobInputs(
            name="retail-agent-eval-set",
            scenario=DataGenerationJobScenario.EVALUATION,
            sources=[
                AgentDataGenerationJobSource(
                    description="Agent definition used to seed QnA generation.",
                    agent_name=agent.name,
                    agent_version=agent.version,
                ),
            ],
            options=SimpleQnADataGenerationJobOptions(
                # Service requires max_samples to be between 15 and 1000.
                max_samples=15,
                # simple_qna requires model_options.
                model_options=DataGenerationModelOptions(model=MODEL_NAME),
            ),
            output_options=DataGenerationJobOutputOptions(name="retail-agent-eval-set"),
        ),
    )

    # 3. Submit and wait for completion.
    poller = project_client.beta.datasets.begin_create_generation_job(job=job)
    while not poller.done():
        print(f"\tstatus=`{poller.status()}`")
        time.sleep(poll_interval_seconds)
    result = poller.result()

    # 4. Resolve the generated dataset.
    output_name = ""
    output_version = ""
    for output in (result.outputs if result is not None else None) or []:
        if isinstance(output, DatasetDataGenerationJobOutput):
            output_name = output.name or ""
            output_version = output.version or ""
            break

    dataset = project_client.datasets.get(name=output_name, version=output_version)
    print(f"Generated dataset: {dataset.name} v{dataset.version} (id: {dataset.id})")
